## Setup

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

/content


## Install YOLO via Ultralytics

In [ ]:
%pip install "ultralytics<=8.3.40" supervision roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.3.40 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.3/112.6 GB disk)


In [ ]:
import supervision as sv

detections = sv.Detections.from_ultralytics(result)

## Fine-tune YOLO5s on custom dataset

In [ ]:
%cd {HOME}

/content


In [ ]:
import kagglehub


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"xaviermoreno","key":"5bde26c03f1c646788ab083adfa97682"}'}

In [ ]:
! mkdir ~/.kaggle

! cp kaggle.json ~/.kaggle/

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

/content/datasets


In [ ]:

# Download latest version
path = kagglehub.dataset_download("jhontroya/dectectra-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/dectectra-dataset


In [ ]:
import shutil, os

# Ruta original (solo lectura)
origen = '/kaggle/input/dectectra-dataset'
# o en Colab:
# origen = '/content/drive/MyDrive/...'

# Ruta destino (editables)
destino = '/content/dataset/dectectra-dataset'

# Copiar todo el contenido
shutil.copytree(origen, destino)


path = "/content/dataset/dectectra-dataset"


'/content/dataset/dectectra-dataset'

In [ ]:
# pendiente de ejecutar
import os

dataset_location = path #"/content/dataset/dectectra-dataset"#/train_data path  # Usa la variable path definida previamente
base_path = os.path.join(dataset_location, "train_data")
img_path = os.path.join(base_path, 'images')
label_path_wrong = os.path.join(base_path, 'labes')
label_path = os.path.join(base_path, 'labels')

# Renombrar carpeta si hace falta
if os.path.exists(label_path_wrong): os.rename(label_path_wrong, label_path)

# Eliminar imágenes sin .txt en train/val/test
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(img_path, split)
    lbl_dir = os.path.join(label_path, split)
    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir): continue
    for img_file in os.listdir(img_dir):
        name, ext = os.path.splitext(img_file)
        if ext.lower() not in ['.jpg', '.jpeg', '.png']: continue
        if not os.path.exists(os.path.join(lbl_dir, name + '.txt')):
            os.remove(os.path.join(img_dir, img_file))
            print(f"🗑️  Eliminado: {img_file} (split: {split})")

# Detectar clases únicas
clases = set()
for split in ['train', 'val', 'test']:
    lbl_dir = os.path.join(label_path, split)
    if not os.path.exists(lbl_dir): continue
    for txt in os.listdir(lbl_dir):
        with open(os.path.join(lbl_dir, txt), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts and parts[0].isdigit():
                    clases.add(int(parts[0]))

# Crear YAML con rutas absolutas
clases = sorted(list(clases))
nc = len(clases)
names = [str(c) for c in clases]

yaml_content = f"""train: {os.path.abspath(os.path.join(img_path, 'train'))}
val: {os.path.abspath(os.path.join(img_path, 'val'))}
test: {os.path.abspath(os.path.join(img_path, 'test'))}

nc: {nc}
names: {names}
"""

yaml_path = os.path.join(base_path, 'dectectra.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\n✅ YAML actualizado con {nc} clases y guardado en: {yaml_path}")


🗑️  Eliminado: 7235.jpg (split: train)
🗑️  Eliminado: 7211.jpg (split: train)
🗑️  Eliminado: 7251.jpg (split: train)
🗑️  Eliminado: 7243.jpg (split: train)
🗑️  Eliminado: 7250.jpg (split: train)
🗑️  Eliminado: 7223.jpg (split: train)
🗑️  Eliminado: 7206.jpg (split: train)
🗑️  Eliminado: 7221.jpg (split: train)
🗑️  Eliminado: 7219.jpg (split: train)
🗑️  Eliminado: 7216.jpg (split: train)
🗑️  Eliminado: 7232.jpg (split: train)
🗑️  Eliminado: 7238.jpg (split: train)
🗑️  Eliminado: 7213.jpg (split: train)
🗑️  Eliminado: 7204.jpg (split: train)
🗑️  Eliminado: 7248.jpg (split: train)
🗑️  Eliminado: 7230.jpg (split: train)
🗑️  Eliminado: 7226.jpg (split: train)
🗑️  Eliminado: 7229.jpg (split: train)
🗑️  Eliminado: 7207.jpg (split: train)
🗑️  Eliminado: 7239.jpg (split: train)
🗑️  Eliminado: 7218.jpg (split: train)
🗑️  Eliminado: 7225.jpg (split: train)
🗑️  Eliminado: 7210.jpg (split: train)
🗑️  Eliminado: 7253.jpg (split: train)
🗑️  Eliminado: 7208.jpg (split: train)
🗑️  Eliminado: 7241.jpg (

## Custom Training

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo11s.pt data={base_path}/dectectra.yaml epochs=300 imgsz=640 plots=True

In [ ]:
!yolo task=detect mode=train model=/content/runs/detect/train/weights/last.pt data={base_path}/dectectra.yaml epochs=10 imgsz=640 plots=True

**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

In [ ]:
!ls {HOME}/runs/detect/train/

args.yaml					     train_batch0.jpg
events.out.tfevents.1744521133.1d5568ab4326.11658.0  train_batch1.jpg
labels_correlogram.jpg				     train_batch2.jpg
labels.jpg					     weights


## Validate fine-tuned model

In [ ]:
!yolo task=detect mode=test model={HOME}/runs/detect/train/weights/best.pt data={base_path}/dectectra.yaml

In [ ]:
import json

json_path = 'runs/detect/val/results.json'  # asegúrate que sea la ruta correcta

with open(json_path) as f:
    data = json.load(f)

print(f"mAP@0.5:      {data['metrics/mAP_0.5']:.4f}")
print(f"mAP@0.5:0.95: {data['metrics/mAP_0.5:0.95']:.4f}")
print(f"Precision:    {data['metrics/precision']:.4f}")
print(f"Recall:       {data['metrics/recall']:.4f}")

In [ ]:
import json

results_path = 'runs/detect/val/results.json'

with open(results_path) as f:
    data = json.load(f)

tp = data['stats/segments/box/total_correct']
total_gt = data['stats/segments/box/total_targets']
fn = total_gt - tp

accuracy = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"TP: {tp}")
print(f"FN: {fn}")
print(f"Accuracy ≈ TP / (TP + FN) = {accuracy:.4f}")


## Inference with custom model

In [ ]:
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.25 source=/content/dataset/dectectra-dataset/train_data/test/images save=True

In [ ]:
!pip install inference

In [ ]:
import os, random, cv2
import supervision as sv
import IPython
import inference

model_id = project.id.split("/")[1] + "/" + dataset.version
model = inference.get_model(model_id, userdata.get('ROBOFLOW_API_KEY'))

# Location of test set images
test_set_loc = dataset.location + "/test/images/"
test_images = os.listdir(test_set_loc)

# Run inference on 4 random test images, or fewer if fewer images are available
for img_name in random.sample(test_images, min(4, len(test_images))):
    print("Running inference on " + img_name)

    # Load image
    image = cv2.imread(os.path.join(test_set_loc, img_name))

    # Perform inference
    results = model.infer(image, confidence=0.4, overlap=30)[0]
    detections = sv.Detections.from_inference(results)

    # Annotate boxes and labels
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    annotated_image = box_annotator.annotate(scene=image, detections=detections)
    annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

    # Display annotated image
    _, ret = cv2.imencode('.jpg', annotated_image)
    i = IPython.display.Image(data=ret)
    IPython.display.display(i)
